# 03 — A Single-Cell Foundation Model for Cell Type Classification

In notebook 02, annotating cell types required a human who already knew the relevant marker genes
for the tissue in question. That doesn't scale: every new tissue, species, or disease context needs
someone who already has that domain knowledge, and you need enough cells per cluster to get a
statistically clean marker gene signal in the first place.

**Single-cell foundation models** take a different approach. Instead of hand-coded marker genes,
a transformer is pretrained on tens of millions of real single cells to learn a general-purpose
representation of "what a cell's expression profile looks like" — similar in spirit to how a language
model learns a general-purpose representation of text before anyone fine-tunes it on a specific
task. We're using [**Geneformer**](https://huggingface.co/ctheodoris/Geneformer)
(Theodoris et al., *Nature* 2023), specifically the smallest available checkpoint
(`Geneformer-V1-10M`, ~10M parameters, pretrained on ~30M cells), which fits comfortably on a
laptop CPU/GPU.

**The question this notebook answers:** if you only have a handful of labeled cells per type — the
realistic scenario when expert annotation is expensive — do Geneformer's pretrained representations
let you classify cell types more accurately than the classical PCA features from notebook 02?

### How Geneformer represents a cell

Geneformer doesn't feed in raw expression values. For each cell, it:

1. Ranks that cell's genes by expression, normalized against each gene's typical expression level
   across a huge reference corpus (so a gene that's *unusually* high for itself, even at modest
   absolute counts, ranks higher than a gene that's merely abundant for every cell type).
2. Feeds the resulting ordered list of gene tokens into a BERT-style transformer, pretrained with a
   masked-gene-prediction objective (mask out some genes in the rank list, predict which gene
   belongs there) — directly analogous to masked-language-modeling in NLP.

The output we care about is the model's hidden representation of the whole cell: a dense vector
that — if pretraining worked — places biologically similar cells near each other, without ever
being told what a "T cell" or "B cell" is.

## 0. Setup

Geneformer isn't on PyPI — it's only distributed from its Hugging Face repo, bundled alongside
several much larger model checkpoints (its V2 models are 100M-300M+ parameters). Run
`scripts/setup_geneformer.sh` from the repo root once before this notebook (it downloads only the
~50MB V1-10M checkpoint + library code via `huggingface_hub`'s `allow_patterns`, skipping the rest):

```bash
bash scripts/setup_geneformer.sh
```

If you haven't run that yet, do it now in a terminal, then come back and run the cells below.

In [ ]:
import os

# On Colab, opening a notebook file directly starts a fresh kernel whose working
# directory is /content, regardless of where (or whether) you previously cloned
# the repo in another notebook/cell. Relative paths like "../data/..." only
# resolve correctly if the kernel's cwd is this repo's notebooks/ directory, so
# make sure of that here before anything else runs. This is a no-op locally.
if os.path.basename(os.getcwd()) != "notebooks":
    for candidate in ("biohack-2026/notebooks", "notebooks"):
        if os.path.isdir(candidate):
            os.chdir(candidate)
            break
print("Working directory:", os.getcwd())

In [ ]:
import pickle
import shutil
import time
from pathlib import Path

import numpy as np
import pandas as pd
import scanpy as sc
import torch
import matplotlib.pyplot as plt
from datasets import load_from_disk
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

GENEFORMER_REPO = Path("../vendor/geneformer_repo")
assert GENEFORMER_REPO.exists(), (
    "Geneformer not found — run `bash scripts/setup_geneformer.sh` from the repo root first."
)

from geneformer import TranscriptomeTokenizer

## 1. Prepare the data

Geneformer needs **raw, unnormalized counts** across the cell's full transcriptome (not the
log-normalized, HVG-subsetted matrix from notebook 02) — it does its own rank-based normalization
internally. We recover that from `adata.raw`, which notebook 01 set before any filtering of genes
or normalization.

We'll also pull in the cell type labels from notebook 02, and subsample to a manageable number of
cells per type so this runs in about a minute — bump `N_PER_TYPE` up (or remove the subsampling
entirely) if you have more time and want embeddings for every cell.

In [ ]:
adata = sc.read_h5ad("../data/pbmc3k_qc.h5ad")
raw = adata.raw.to_adata()  # full gene panel, raw counts, only cell-level QC filtering applied

labels = pd.read_csv("../data/pbmc3k_cell_type_labels.csv").set_index("cell_id")
raw = raw[raw.obs_names.isin(labels.index)].copy()
raw.obs["cell_type"] = labels.loc[raw.obs_names, "cell_type"].values

N_PER_TYPE = 120
rng = np.random.default_rng(0)
keep = []
for ct, grp in raw.obs.groupby("cell_type"):
    idx = grp.index.tolist()
    n = min(N_PER_TYPE, len(idx))
    keep.extend(rng.choice(idx, size=n, replace=False).tolist())
raw = raw[keep].copy()

print(raw.shape)
raw.obs["cell_type"].value_counts()

## 2. Map gene symbols to Ensembl IDs

Geneformer's vocabulary is keyed by Ensembl gene ID, not gene symbol. Conveniently, the package
ships its own symbol-to-Ensembl-ID dictionary (the same one it used during pretraining), so this is
a pure local lookup.

Not every gene symbol in our data will have a match: Geneformer's vocabulary covers ~25-40K
protein-coding and well-characterized genes, while PBMC3k's gene panel includes thousands of
obscure, non-coding, or poorly annotated loci that were never going to be in scope for a model
trained on curated single-cell atlases. Seeing roughly half your genes drop out here is expected,
not a bug.

In [ ]:
with open(GENEFORMER_REPO / "geneformer/gene_dictionaries_30m/gene_name_id_dict_gc30M.pkl", "rb") as f:
    symbol_to_ensembl = pickle.load(f)

raw.var["ensembl_id"] = [symbol_to_ensembl.get(g) for g in raw.var_names]
raw.obs["n_counts"] = np.asarray(raw.X.sum(axis=1)).flatten()
raw.obs["cell_id"] = raw.obs_names

n_mapped = raw.var["ensembl_id"].notna().sum()
print(f"mapped {n_mapped} / {raw.n_vars} genes to Ensembl IDs")

## 3. Tokenize

`TranscriptomeTokenizer` does the rank-value encoding described above: for each cell, it normalizes
expression against the gene median dictionary from Geneformer's pretraining corpus, ranks genes
accordingly, and writes out a token sequence per cell as a Hugging Face `datasets.Dataset`.

In [ ]:
work_dir = Path("/tmp/geneformer_work")
shutil.rmtree(work_dir, ignore_errors=True)
(work_dir / "in").mkdir(parents=True)
raw.write(work_dir / "in" / "cells.h5ad")

tokenizer = TranscriptomeTokenizer(
    custom_attr_name_dict={"cell_id": "cell_id"},
    nproc=1,
    model_version="V1",     # the 10M-parameter checkpoint was pretrained as a "V1" model
    special_token=False,    # V1 models predate the <cls>/<eos> special tokens V2 uses
    model_input_size=2048,  # V1's context length
)
tokenizer.tokenize_data(work_dir / "in", work_dir / "out", "cells", file_format="h5ad")

dataset = load_from_disk(work_dir / "out" / "cells.dataset")
print(f"tokenized {len(dataset)} cells")
print("example token sequence length:", dataset[0]["length"])

## 4. Load the model and extract cell embeddings

A note on the implementation here: Geneformer ships a convenience class for this (`EmbExtractor`),
but as of this writing its embedding-extraction code path has `device="cuda"` hardcoded with no CPU
or Apple Silicon (MPS) fallback — it'll crash with `AssertionError: Torch not compiled with CUDA
enabled` on any machine without an Nvidia GPU, which is most laptops at a hackathon. This is a good
reminder that real research tooling has real rough edges. Instead, we load the underlying
Hugging Face model directly and write our own short, transparent, device-agnostic forward pass —
which also makes it clearer what's actually happening: we mean-pool the model's last hidden layer
across each cell's (unpadded) token sequence to get one vector per cell.

One efficiency detail: cells have wildly different numbers of expressed genes, so token sequences
range from a few hundred to ~2,000 tokens. Batching cells of very different lengths together forces
heavy padding (and on Apple's MPS backend, batches with extreme padding can stall badly). Sorting by
sequence length before batching — then unsorting the results back to the original order — fixes
both problems and is a generally useful trick whenever you're batching variable-length sequences.

In [ ]:
device = torch.device(
    "mps" if torch.backends.mps.is_available()
    else "cuda" if torch.cuda.is_available()
    else "cpu"
)
print("using device:", device)

from transformers import BertForMaskedLM

model = BertForMaskedLM.from_pretrained(
    GENEFORMER_REPO / "Geneformer-V1-10M", output_hidden_states=True
)
model.eval().to(device)

with open(GENEFORMER_REPO / "geneformer/gene_dictionaries_30m/token_dictionary_gc30M.pkl", "rb") as f:
    token_dictionary = pickle.load(f)
pad_token_id = token_dictionary["<pad>"]

In [ ]:
order = sorted(range(len(dataset)), key=lambda i: dataset[i]["length"])
dataset_sorted = dataset.select(order)

batch_size = 16
embeddings_sorted = []

t0 = time.time()
with torch.no_grad():
    for i in range(0, len(dataset_sorted), batch_size):
        batch = dataset_sorted[i : i + batch_size]
        lengths = batch["length"]
        max_len = max(lengths)

        input_ids = torch.tensor(
            [seq + [pad_token_id] * (max_len - len(seq)) for seq in batch["input_ids"]],
            device=device,
        )
        attention_mask = torch.tensor(
            [[1] * l + [0] * (max_len - l) for l in lengths], device=device
        )

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        hidden = outputs.hidden_states[-1]               # (batch, seq_len, hidden_dim)
        mask = attention_mask.unsqueeze(-1).bool()
        mean_pooled = (hidden * mask).sum(dim=1) / mask.sum(dim=1)
        embeddings_sorted.append(mean_pooled.cpu().numpy())

embeddings_sorted = np.concatenate(embeddings_sorted)
inverse_order = np.argsort(order)
embeddings = embeddings_sorted[inverse_order]            # back to dataset's original cell order
cell_ids = dataset["cell_id"]

print(f"extracted {embeddings.shape[0]} embeddings of dimension {embeddings.shape[1]} "
      f"in {time.time() - t0:.1f}s")

np.save("../data/pbmc3k_geneformer_embeddings.npy", embeddings)
pd.Series(cell_ids).to_csv("../data/pbmc3k_geneformer_cell_ids.csv", index=False)

## 5. Visualize: does the foundation model's embedding space separate cell types?

Geneformer never saw our `cell_type` labels — they come entirely from notebook 02's classical
pipeline. If the embedding space below visibly separates by color, that's a real signal the
pretrained model learned biologically meaningful structure with zero supervision on this dataset.

In [ ]:
import anndata as ad

emb_adata = ad.AnnData(
    X=embeddings,
    obs=pd.DataFrame({"cell_type": pd.Categorical(
        labels.loc[cell_ids, "cell_type"].values
    )}, index=cell_ids),
)
sc.pp.neighbors(emb_adata, use_rep="X")
sc.tl.umap(emb_adata)
sc.pl.umap(emb_adata, color="cell_type", title="Geneformer embeddings (UMAP)")

## 6. The real test: few-shot cell type classification

A UMAP plot is suggestive but qualitative. Here's a concrete, quantitative comparison: train a
simple logistic regression classifier to predict `cell_type` from features, using only a handful of
labeled examples per class — then check accuracy on the rest. We run this for both:

- **Classical features**: the PCA components from notebook 02's pipeline.
- **Geneformer embeddings**: this notebook's foundation model representations.

This simulates the most common real constraint in biology: unlabeled cells are cheap (sequence
anything), but *expert-annotated* cells are expensive and scarce. A representation that needs fewer
labeled examples to reach good accuracy is directly valuable.

In [ ]:
pca_features = pd.read_csv("../data/pbmc3k_pca_features.csv", index_col=0)
gf_features = pd.DataFrame(embeddings, index=cell_ids)

common_cells = gf_features.index.intersection(pca_features.index).intersection(labels.index)
y = labels.loc[common_cells, "cell_type"].values
X_pca = pca_features.loc[common_cells].values
X_gf = gf_features.loc[common_cells].values

print(f"{len(common_cells)} cells with both feature sets and labels")

In [ ]:
def few_shot_accuracy(X, y, n_shots, n_seeds=15):
    accs = []
    for seed in range(n_seeds):
        r = np.random.default_rng(seed * 100 + n_shots)
        train_idx = []
        for cls in np.unique(y):
            cls_idx = np.where(y == cls)[0]
            if len(cls_idx) <= n_shots:
                continue
            train_idx.extend(r.choice(cls_idx, size=n_shots, replace=False))
        train_idx = np.array(train_idx)
        test_idx = np.array([i for i in range(len(y)) if i not in set(train_idx)])

        scaler = StandardScaler().fit(X[train_idx])
        X_train, X_test = scaler.transform(X[train_idx]), scaler.transform(X[test_idx])

        clf = LogisticRegression(max_iter=2000)
        clf.fit(X_train, y[train_idx])
        accs.append(clf.score(X_test, y[test_idx]))
    return np.mean(accs), np.std(accs)

shot_counts = [1, 3, 5, 10, 20]
results = []
for n_shots in shot_counts:
    pca_mean, pca_std = few_shot_accuracy(X_pca, y, n_shots)
    gf_mean, gf_std = few_shot_accuracy(X_gf, y, n_shots)
    results.append((n_shots, pca_mean, pca_std, gf_mean, gf_std))
    print(f"{n_shots:>2} shots/class -- PCA: {pca_mean:.3f} +/- {pca_std:.3f}  "
          f"Geneformer: {gf_mean:.3f} +/- {gf_std:.3f}")

In [ ]:
results_df = pd.DataFrame(
    results, columns=["n_shots", "pca_mean", "pca_std", "gf_mean", "gf_std"]
)

plt.figure(figsize=(7, 5))
plt.errorbar(results_df.n_shots, results_df.pca_mean, yerr=results_df.pca_std,
             marker="o", label="Classical PCA features", capsize=3)
plt.errorbar(results_df.n_shots, results_df.gf_mean, yerr=results_df.gf_std,
             marker="o", label="Geneformer embeddings", capsize=3)
plt.xlabel("labeled examples per cell type (\"shots\")")
plt.ylabel("held-out classification accuracy")
plt.title("Few-shot cell type classification: foundation model vs. classical features")
plt.legend()
plt.ylim(0, 1.05)
plt.grid(alpha=0.3)
plt.show()

**Stop and look at the curve.** In our run, Geneformer embeddings reached roughly 80% accuracy
with a *single* labeled cell per type, while classical PCA features needed around 10-20 labeled
cells per type to catch up. That gap is the entire value proposition of a foundation model in this
setting: pretraining on ~30M cells gave the model a strong prior about what distinguishes cell
types *before it ever saw this dataset*, so it needs far less task-specific labeled data to do well.

The two methods converge as labeled data increases — with enough labels, the classical features
eventually catch up. That's the expected, realistic story: foundation models are most valuable
exactly when labels are scarce, not a strictly-better-in-all-cases replacement for classical
analysis.

Some questions worth discussing with your team:

- Why might the gap be largest at very low shot counts and shrink as shots increase?
- We used a 10M-parameter checkpoint pretrained on ~30M cells. Geneformer's largest public
  checkpoint has 316M parameters and was pretrained on ~104M cells. What would you predict happens
  to this curve with a bigger model?
- We froze the embeddings and only trained a linear classifier on top. What might change if we
  fine-tuned the whole model instead (and what would that cost in compute/time)?
- This dataset's classes are all well-separated, common PBMC types. Would you expect the same size
  of advantage for a harder problem — e.g., distinguishing two closely related T cell subtypes?

## Recap

- A foundation model pretrained on millions of cells can encode a strong, general-purpose prior
  about cell biology — usable for a new dataset's classification task without any task-specific
  pretraining, just a lightweight classifier on top of its embeddings.
- That prior is most valuable exactly when labeled data is scarce, which is the normal situation in
  real biological annotation work.
- Real ML/bio tooling has rough edges (the hardcoded-CUDA issue here) that you'll hit constantly in
  practice — reading the library's source when something breaks is a normal and necessary part of
  the work, not a sign you're doing something wrong.

This closes the loop on the whole pipeline: raw counts -> QC -> classical clustering and manual
annotation -> foundation model embeddings -> a quantitative, reproducible argument for when and why
the foundation model is worth using.